# Longformer Fine-Tuning -- v3 (High-LR Robustness Check)

**Purpose:** the LSTM comparator (notebook 06) beat the Longformer v2 result on
every metric. Before accepting that as a real finding, this notebook checks the
obvious confound: was the Longformer's `2e-5` fine-tuning LR too low to converge
its randomly-initialized classification head in only ~85-100 steps, unlike the
LSTM's `1e-3` (appropriate for training entirely from scratch)?

**Only change from notebook 04: `learning_rate` 2e-5 -> 1e-4 (5x). Everything
else identical** (same folds, same steps, same epochs) -- a clean, single-variable
test.

**Result: the LR hypothesis was REFUTED.** ROC-AUC barely moved (0.827 -> 0.835),
PR-AUC got worse (0.511 -> 0.474). The LSTM's advantage over the Longformer is a
genuine architecture-level finding (catalogue angle A4, negative result), not an
LR artifact. See `supervisor_notes/note_2026-07-14_a4-lstm-vs-longformer-finding.md`.

**Reporting convention:** v2 (notebook 04) is the PRIMARY reported result. This
v3 run is a robustness check / control, not a replacement.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/thesis'

## Fine-tune with 5x higher LR (1e-4), same folds/steps as v2

In [ ]:
"""
Fine-tune the Phase 3 pretrained Longformer for binary BOLA/BFLA sequence
classification, across all LOAO folds, with N_RUNS independent runs per
fold for variance bounds. Identical to notebook 04 except learning_rate.

Saves per-fold-per-run predictions to:
  {BASE}/results/longformer_finetune_v3_highlr/fold_<app>_run<n>.json
"""

import os
import json
import glob
import random
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    LongformerForSequenceClassification, LongformerTokenizer,
    TrainingArguments, Trainer
)
from sklearn.metrics import precision_recall_curve, roc_curve, f1_score

BASE = "/content/drive/MyDrive/thesis"
PRETRAINED_PATH = f"{BASE}/models/longformer/phase3_final"
FOLDS_DIR = f"{BASE}/data/loao_folds_v2"
RESULTS_DIR = f"{BASE}/results/longformer_finetune_v3_highlr"  # new dir -- keeps v2 results intact for comparison
os.makedirs(RESULTS_DIR, exist_ok=True)

MAX_LENGTH_FT = 256    # generous ceiling for 2-request pairs (actual sequences are ~30-60 subword tokens)
N_RUNS = 3             # proposal's three-independent-runs-per-configuration commitment
                        # SET TO 1 FIRST to pilot on a single fold before committing to the full 48 runs

tokenizer = LongformerTokenizer.from_pretrained(PRETRAINED_PATH)


class BinaryClassDataset(Dataset):
    def __init__(self, records, tokenizer, max_length):
        self.sequences = [r["sequence"] for r in records]
        self.labels = [r["label"] for r in records]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.sequences[idx],
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"][0]
        attention_mask = enc["attention_mask"][0]
        global_attention_mask = torch.zeros_like(attention_mask)
        global_attention_mask[0] = 1  # global attention on first token (pooling token)
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "global_attention_mask": global_attention_mask,
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def collate_fn(batch):
    max_len = max(item["input_ids"].size(0) for item in batch)
    pad_id = tokenizer.pad_token_id

    def pad(tensor, pad_value):
        pad_len = max_len - tensor.size(0)
        if pad_len == 0:
            return tensor
        return torch.cat([tensor, torch.full((pad_len,), pad_value, dtype=tensor.dtype)])

    input_ids = torch.stack([pad(item["input_ids"], pad_id) for item in batch])
    attention_mask = torch.stack([pad(item["attention_mask"], 0) for item in batch])
    global_attention_mask = torch.stack([pad(item["global_attention_mask"], 0) for item in batch])
    labels = torch.stack([item["labels"] for item in batch])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "global_attention_mask": global_attention_mask,
        "labels": labels,
    }


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]


def train_and_eval_fold(fold_name, run_idx, seed):
    random.seed(seed)
    torch.manual_seed(seed)

    fold_dir = f"{FOLDS_DIR}/fold_{fold_name}"
    train_records = load_jsonl(f"{fold_dir}/train.jsonl")
    eval_records = load_jsonl(f"{fold_dir}/eval.jsonl")

    train_dataset = BinaryClassDataset(train_records, tokenizer, MAX_LENGTH_FT)
    eval_dataset = BinaryClassDataset(eval_records, tokenizer, MAX_LENGTH_FT)

    model = LongformerForSequenceClassification.from_pretrained(
        PRETRAINED_PATH, num_labels=2
    )

    training_args = TrainingArguments(
        output_dir=f"/tmp/ft_{fold_name}_run{run_idx}",  # scratch, not saved to Drive
        num_train_epochs=5,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=1,
        per_device_eval_batch_size=8,
        learning_rate=1e-4,   # was 2e-5 in notebook 04 -- testing whether the
                               # from-scratch classifier head needs a higher LR
                               # than the fine-tuning-rate default to converge
                               # in ~85-100 steps (same issue the LSTM's 1e-3
                               # solved for its from-scratch parameters)
        logging_steps=10,
        save_strategy="no",       # don't checkpoint -- we only need final predictions
        eval_strategy="no",       # we run eval manually below
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=collate_fn,
    )
    trainer.train()

    # Manual eval: get raw logits -> softmax probabilities for the attack class
    model.eval()
    probs, true_labels = [], []
    with torch.no_grad():
        for i in range(len(eval_dataset)):
            item = collate_fn([eval_dataset[i]])
            item = {k: v.to(model.device) for k, v in item.items() if k != "labels"}
            logits = model(**item).logits
            p = torch.softmax(logits, dim=-1)[0, 1].item()  # P(attack)
            probs.append(p)
            true_labels.append(eval_dataset.labels[i])

    result = {
        "fold": fold_name,
        "run": run_idx,
        "seed": seed,
        "probs": probs,
        "labels": true_labels,
        "n_train": len(train_dataset),
        "n_eval": len(eval_dataset),
    }

    with open(f"{RESULTS_DIR}/fold_{fold_name}_run{run_idx}.json", "w") as f:
        json.dump(result, f)

    del model, trainer
    torch.cuda.empty_cache()

    return result


def main():
    fold_names = sorted(
        d.replace("fold_", "") for d in os.listdir(FOLDS_DIR) if d.startswith("fold_")
    )
    print(f"Found {len(fold_names)} folds: {fold_names}\n")

    all_results = []
    for fold_name in fold_names:
        for run_idx in range(N_RUNS):
            seed = 42 + run_idx
            print(f"--- Fold: {fold_name}, run {run_idx+1}/{N_RUNS} (seed={seed}) ---")
            result = train_and_eval_fold(fold_name, run_idx, seed)
            n_pos = sum(result["labels"])
            n_neg = len(result["labels"]) - n_pos
            print(f"  eval: {n_pos} pos / {n_neg} neg, mean prob (attack class)={np.mean(result['probs']):.3f}")
            all_results.append(result)

    print(f"\nAll {len(all_results)} fold/run results saved to {RESULTS_DIR}")
    print("Run the separate aggregation script next to compute pooled precision/recall/F1/FPR.")


if __name__ == "__main__":
    main()


## Aggregate v3 results

In [ ]:
"""
Aggregate all 48 fold/run prediction files into pooled metrics.

Run in Colab after the fine-tuning cell above completes.
"""

import json
import glob
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, roc_curve

BASE = "/content/drive/MyDrive/thesis"
RESULTS_DIR = f"{BASE}/results/longformer_finetune_v3_highlr"


def load_all_results():
    files = sorted(glob.glob(f"{RESULTS_DIR}/fold_*_run*.json"))
    results = []
    for f in files:
        with open(f) as fh:
            results.append(json.load(fh))
    return results


def main():
    results = load_all_results()
    print(f"Loaded {len(results)} fold/run result files\n")

    all_probs, all_labels = [], []
    for r in results:
        all_probs.extend(r["probs"])
        all_labels.extend(r["labels"])
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    print(f"Pooled eval set: {len(all_labels)} predictions "
          f"({int(all_labels.sum())} positive / {int(len(all_labels) - all_labels.sum())} negative)\n")

    pos_probs = all_probs[all_labels == 1]
    neg_probs = all_probs[all_labels == 0]
    print(f"Positive-class prob:  mean={pos_probs.mean():.3f}, median={np.median(pos_probs):.3f}, std={pos_probs.std():.3f}")
    print(f"Negative-class prob:  mean={neg_probs.mean():.3f}, median={np.median(neg_probs):.3f}, std={neg_probs.std():.3f}")

    auc_roc = roc_auc_score(all_labels, all_probs)
    auc_pr = average_precision_score(all_labels, all_probs)
    print(f"\nPooled ROC-AUC: {auc_roc:.3f}  (0.5 = random, 1.0 = perfect)")
    print(f"Pooled PR-AUC:  {auc_pr:.3f}  (baseline = positive rate = {all_labels.mean():.3f})\n")

    precisions, recalls, thresholds = precision_recall_curve(all_labels, all_probs)
    fpr, tpr, roc_thresholds = roc_curve(all_labels, all_probs)

    print("Recall at various FPR operating points (pooled):")
    for target_fpr in [0.05, 0.10, 0.20, 0.30]:
        idx = np.searchsorted(fpr, target_fpr)
        idx = min(idx, len(tpr) - 1)
        print(f"  FPR <= {target_fpr:.2f}: recall = {tpr[idx]:.3f} (threshold = {roc_thresholds[idx]:.3f})")

    preds_at_half = (all_probs >= 0.5).astype(int)
    tp = int(((preds_at_half == 1) & (all_labels == 1)).sum())
    fp = int(((preds_at_half == 1) & (all_labels == 0)).sum())
    fn = int(((preds_at_half == 0) & (all_labels == 1)).sum())
    tn = int(((preds_at_half == 0) & (all_labels == 0)).sum())
    precision_half = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall_half = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr_half = fp / (fp + tn) if (fp + tn) > 0 else 0
    f1_half = 2 * precision_half * recall_half / (precision_half + recall_half) if (precision_half + recall_half) > 0 else 0
    print(f"\nAt threshold=0.5: precision={precision_half:.3f}, recall={recall_half:.3f}, "
          f"FPR={fpr_half:.3f}, F1={f1_half:.3f}  (TP={tp}, FP={fp}, FN={fn}, TN={tn})\n")

    fold_names = sorted(set(r["fold"] for r in results))
    print("Per-fold recall (mean +/- std across 3 runs), at threshold=0.5:")
    for fold in fold_names:
        fold_runs = [r for r in results if r["fold"] == fold]
        fold_recalls = []
        for r in fold_runs:
            probs = np.array(r["probs"])
            labels = np.array(r["labels"])
            pos_mask = labels == 1
            if pos_mask.sum() == 0:
                continue
            preds = (probs >= 0.5).astype(int)
            recall = (preds[pos_mask] == 1).mean()
            fold_recalls.append(recall)
        if fold_recalls:
            print(f"  {fold:25s}: {np.mean(fold_recalls):.3f} +/- {np.std(fold_recalls):.3f}  (n_runs={len(fold_recalls)})")

    np.savez(f"{RESULTS_DIR}/pooled_predictions.npz", probs=all_probs, labels=all_labels)
    print(f"\nPooled predictions saved to {RESULTS_DIR}/pooled_predictions.npz")


if __name__ == "__main__":
    main()
